In [31]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder,OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score,accuracy_score
from sklearn.metrics import mean_absolute_error, mean_squared_error


In [2]:
df=pd.read_csv("D:\ML\datasets\CAR DETAILS FROM CAR DEKHO.csv")

In [3]:
df.shape

(4340, 8)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4340 entries, 0 to 4339
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   name           4340 non-null   object
 1   year           4340 non-null   int64 
 2   selling_price  4340 non-null   int64 
 3   km_driven      4340 non-null   int64 
 4   fuel           4340 non-null   object
 5   seller_type    4340 non-null   object
 6   transmission   4340 non-null   object
 7   owner          4340 non-null   object
dtypes: int64(3), object(5)
memory usage: 271.4+ KB


In [5]:
df.describe()

,year,selling_price,km_driven
count,4340.000000,4.340000e+03,4340.000000
mean,2013.090783,5.041273e+05,66215.777419
std,4.215344,5.785487e+05,46644.102194
min,1992.000000,2.000000e+04,1.000000
25%,2011.000000,2.087498e+05,35000.000000
50%,2014.000000,3.500000e+05,60000.000000
75%,2016.000000,6.000000e+05,90000.000000
max,2020.000000,8.900000e+06,806599.000000


In [6]:
df.isnull().sum()

name             0
year             0
selling_price    0
km_driven        0
fuel             0
seller_type      0
transmission     0
owner            0
dtype: int64

In [7]:
df.head()

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner
0,Maruti 800 AC,2007,60000,70000,Petrol,Individual,Manual,First Owner
1,Maruti Wagon R LXI Minor,2007,135000,50000,Petrol,Individual,Manual,First Owner
2,Hyundai Verna 1.6 SX,2012,600000,100000,Diesel,Individual,Manual,First Owner
3,Datsun RediGO T Option,2017,250000,46000,Petrol,Individual,Manual,First Owner
4,Honda Amaze VX i-DTEC,2014,450000,141000,Diesel,Individual,Manual,Second Owner


In [8]:
df['name'].nunique()

1491

In [9]:
df['brand']=df['name'].str.split().str[0]

In [10]:
df['brand']

0        Maruti
1        Maruti
2       Hyundai
3        Datsun
4         Honda
         ...   
4335    Hyundai
4336    Hyundai
4337     Maruti
4338    Hyundai
4339    Renault
Name: brand, Length: 4340, dtype: object

In [11]:
df.head()

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,brand
0,Maruti 800 AC,2007,60000,70000,Petrol,Individual,Manual,First Owner,Maruti
1,Maruti Wagon R LXI Minor,2007,135000,50000,Petrol,Individual,Manual,First Owner,Maruti
2,Hyundai Verna 1.6 SX,2012,600000,100000,Diesel,Individual,Manual,First Owner,Hyundai
3,Datsun RediGO T Option,2017,250000,46000,Petrol,Individual,Manual,First Owner,Datsun
4,Honda Amaze VX i-DTEC,2014,450000,141000,Diesel,Individual,Manual,Second Owner,Honda


In [12]:
x=df.drop(columns=['name','selling_price'])
y=df['selling_price']

In [13]:
xtrain,xtest,ytrain,ytest=train_test_split(x,y,train_size=0.8,random_state=103)

In [14]:
cat_col=xtrain.select_dtypes(include='object').columns
num_col=xtrain.select_dtypes(include='number').columns

In [15]:
print(cat_col)

Index(['fuel', 'seller_type', 'transmission', 'owner', 'brand'], dtype='object')


In [16]:
one_hot=['fuel', 'seller_type', 'transmission', 'brand']
ordi_nal=['owner']

In [17]:
df['owner'].unique()

array(['First Owner', 'Second Owner', 'Fourth & Above Owner',
       'Third Owner', 'Test Drive Car'], dtype=object)

In [18]:
preprocessor=ColumnTransformer(
   transformers=[
       ('One_Hot_encoding',OneHotEncoder(handle_unknown='ignore'),one_hot),
       ('Ordinal_encoding',OrdinalEncoder(categories=[[
           'First Owner', 
           'Second Owner', 
           'Fourth & Above Owner',
            'Third Owner', 
            'Test Drive Car'    
       ]]),ordi_nal)
   ],
    remainder='passthrough'
)

In [34]:
model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(
        n_estimators=200,
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features='sqrt',
        random_state=42
    ))
])

In [35]:
model.fit(xtrain,ytrain)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('One_Hot_encoding', ...), ('Ordinal_encoding', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [36]:
train_ypred=model.predict(xtrain)
test_ypred=model.predict(xtest)


In [37]:
r2_Score_Train=r2_score(ytrain,train_ypred)
r2_Score_test=r2_score(ytest,test_ypred)
print(f"R2_Score of Train Data:{r2_Score_Train} \nR2_Score Of Test Data: {r2_Score_test}")
print("Diffrence:",r2_Score_Train-r2_Score_test)


R2_Score of Train Data:0.8074685965159354 
R2_Score Of Test Data: 0.7453474867724716
Diffrence: 0.06212110974346374


In [38]:
mae = mean_absolute_error(ytest, test_ypred)
mse = mean_squared_error(ytest, test_ypred)
rmse = mean_squared_error(ytest, test_ypred) ** 0.5

print("MAE :", mae)
print("MSE :", mse)
print("RMSE:", rmse)

MAE : 160294.0588806064
MSE : 91019573818.05377
RMSE: 301694.50412305124


In [39]:
feature_names = model.named_steps['preprocessor'].get_feature_names_out()

importance = pd.DataFrame({
    'Feature': feature_names,
    'Importance': model.named_steps['model'].feature_importances_
})

importance = importance.sort_values(
    by='Importance',
    ascending=False
)

print(importance.head(10))

                                     Feature  Importance
40                           remainder__year    0.156723
8   One_Hot_encoding__transmission_Automatic    0.148047
9      One_Hot_encoding__transmission_Manual    0.136601
12               One_Hot_encoding__brand_BMW    0.088210
41                      remainder__km_driven    0.088128
29     One_Hot_encoding__brand_Mercedes-Benz    0.067908
1              One_Hot_encoding__fuel_Diesel    0.058870
4              One_Hot_encoding__fuel_Petrol    0.056974
11              One_Hot_encoding__brand_Audi    0.052401
39                   Ordinal_encoding__owner    0.023776
